In [1]:
import json
import os
import time
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from imggen import image_generator
from strgen import Story_content_generator

# Fix Windows console encoding for emojis (skip in environments where not supported)
import sys
if sys.platform == 'win32' and hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding='utf-8')

# Set credentials file if not already set in your environment
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"F:\Users\sarat\Documents\imagetostory\imgstr-95ee242a4e84.json"

# Vertex AI settings
PROJECT_ID = "imgstr"
LOCATION = "global"  # Required for gemini-3-pro-image-preview model

# NOTE:
# `imggen.image_generator()` now auto-normalizes CHARACTER images to a consistent 1024x1024 canvas
# (keeps pages + cover untouched). This makes multi-character scene composition more consistent.

In [2]:
# Set environment variable
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"F:\Users\sarat\Documents\imagetostory\imgstr-95ee242a4e84.json"

os.makedirs("generated", exist_ok=True)
os.makedirs("generated_images", exist_ok=True)
print("✅ Setup complete!")

# Initialize timing trackers
timing = {
    'story_gen': 0,
    'parse': 0,
    'characters': 0,
    'book_cover': 0,
    'pages': 0,
    'html': 0
}

# ============================================================================
# STEP 1: Generate Story JSON
# ============================================================================
print("\n📖 Generating story JSON...")
print("="*70)
story_gen_start = time.time()
json_str = Story_content_generator(
    story_prompt="i gave you image of babu and i want you to create an urban city adventure story lik jamesbond with babu and jr.ntr",
    image_paths=["input_images/babu.jpeg"],
    output_dir="generated"  # ✅ This now works!
)
timing['story_gen'] = time.time() - story_gen_start

print(f"\n⏱️  Story JSON generation completed in {timing['story_gen']:.1f} seconds")

# ============================================================================
# STEP 2: Parse JSON
# ============================================================================
def clean_json_output(json_str):
    """Remove markdown code blocks from JSON string if present."""
    json_str_clean = json_str.strip()
    
    # Remove markdown code blocks
    if json_str_clean.startswith("```"):
        # Remove ```json from start
        json_str_clean = json_str_clean.split("```", 1)[1]
        if json_str_clean.startswith("json"):
            json_str_clean = json_str_clean[4:]
        # Remove ``` from end
        json_str_clean = json_str_clean.rsplit("```", 1)[0]
        json_str_clean = json_str_clean.strip()
    
    return json_str_clean

# Clean and parse
parse_start = time.time()
json_str_clean = clean_json_output(json_str)
story = json.loads(json_str_clean)

# Generate timestamp for filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
timestamped_filename = f"story_data_{timestamp}.json"

# Save JSON with timestamp
with open(timestamped_filename, "w", encoding="utf-8") as f:
    json.dump(story, f, indent=2, ensure_ascii=False)

# Also save as story_data.json (for HTML generation compatibility)
with open("story_data.json", "w", encoding="utf-8") as f:
    json.dump(story, f, indent=2, ensure_ascii=False)

timing['parse'] = time.time() - parse_start

print("✅ Story generated!")
print(f"\n📊 Summary:")
print(f"  - Book: {story['book']['title']}")
print(f"  - Characters: {1 + len(story['characters'].get('supporting_characters', []))}")
print(f"  - Pages: {len(story['pages'])}")
print(f"\n💾 Saved to:")
print(f"  - {timestamped_filename} (timestamped)")
print(f"  - story_data.json (latest, for HTML generation)")
print(f"⏱️  JSON parsing & saving: {timing['parse']:.2f} seconds")


✅ Setup complete!

📖 Generating story JSON...


INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-pro-preview:generateContent "HTTP/1.1 200 OK"



⏱️  Story JSON generation completed in 54.9 seconds
✅ Story generated!

📊 Summary:
  - Book: Babu's Midnight Mission
  - Characters: 2
  - Pages: 10

💾 Saved to:
  - story_data_20251229_222500.json (timestamped)
  - story_data.json (latest, for HTML generation)
⏱️  JSON parsing & saving: 0.03 seconds


In [3]:
story

{'characters': {'main_character': {'name': 'Babu',
   'description': 'Top secret agent Babu, dressed in a sleek black tuxedo with a crisp white shirt and black bow tie. He has a clean, sharp spy hairstyle and wears a high-tech watch.',
   'input_images': ['input_images/babu.jpeg'],
   'output_image': 'generated/babu_spy_sheet.png',
   'prompt': 'Turn this person into a secret agent spy. Keep the person exactly as shown in the reference image with 100% identical facial features, bone structure, skin tone, and appearance. Only change: costume to a sleek black tuxedo with a white dress shirt and black bow tie, and hairstyle to a neat, professional spy look. Anatomically correct proportions, realistic human scale. Edge-to-edge composition, NO borders, seamless neutral background. Full body pose, face directly toward camera, both eyes visible. Cinematic portrait, dramatic lighting, hyper-realistic, natural skin texture, sharp focus on eyes, photorealistic, ultra-detailed, 8K resolution.'},


In [23]:
from google import genai
from google.genai import types
from pathlib import Path
import base64
from io import BytesIO
from PIL import Image

def _resolve_path(p: str | Path) -> Path:
    """Resolve paths relative to the notebook/kernel working directory."""
    p = Path(p)
    return p if p.is_absolute() else (Path.cwd() / p)


def generate_image(task, *, max_retries: int = 3, retry_sleep_s: float = 2.0) -> str:
    """Generate an image for a task and return the saved output path.

    Raises a clear error if any required input image is missing or if the
    model doesn't return an image after retries (prevents confusing downstream
    FileNotFoundError like generated_images/page_1.png).
    """
    import logging
    import time

    # Suppress noisy INFO loggers for cleaner output
    logging.getLogger("google_genai").setLevel(logging.WARNING)
    logging.getLogger("httpx").setLevel(logging.WARNING)

    # Vertex AI config: try to infer PROJECT_ID/LOCATION if you didn't set them earlier
    import os
    try:
        from google.auth import default as google_auth_default
    except Exception:
        google_auth_default = None

    project_id = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCLOUD_PROJECT")
    if not project_id and google_auth_default is not None:
        try:
            _, project_id = google_auth_default()
        except Exception:
            project_id = None

    location = globals().get("LOCATION") or os.environ.get("GOOGLE_CLOUD_LOCATION") or os.environ.get("VERTEX_LOCATION") or "global"

    if not project_id:
        raise RuntimeError(
            "PROJECT_ID is not set and couldn't be inferred. "
            "Set PROJECT_ID='your-gcp-project-id' (and optionally LOCATION='us-central1') and re-run."
        )

    client = genai.Client(
        vertexai=True,
        project=project_id,
        location=location,
    )

    # Load input images (with robust path resolution + existence checks)
    input_images = []
    input_paths_resolved: list[str] = []

    for img_path in task.get("input_images", []):
        p = _resolve_path(img_path)
        input_paths_resolved.append(str(p))
        if not p.exists():
            raise FileNotFoundError(
                f"Input image not found: '{img_path}' (resolved to '{p}'). "
                f"Current working dir: '{Path.cwd()}'"
            )
        input_images.append(Image.open(p))

    text_input = task["prompt"]

    out_path = _resolve_path(task["output_image"])
    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Print full prompt and input image info for transparency
    print("=" * 60)
    print(f"🖼️  Generating image for: {task.get('name', 'Unknown')}")
    print(f"Prompt:\n{text_input}\n")
    print(
        "Input images: "
        + (", ".join(task.get("input_images", [])) if task.get("input_images") else "(none)")
    )
    if input_paths_resolved:
        print(f"Resolved inputs: {', '.join(input_paths_resolved)}")
    print(f"Output image: {task['output_image']} (resolved: {out_path})")
    print("=" * 60)

    # SAFEST approach: Set all categories to BLOCK_NONE
    safety_settings = [
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
    ]

    # Resolution and Aspect Ratio settings
    aspect_ratio = "1:1"  # Options: "1:1", "3:4", "4:3", "16:9", etc.
    resolution = "2K"  # Options: "1K", "2K" - Using 2K for hyper-realistic detail

    last_text_debug = ""

    for attempt in range(1, max_retries + 1):
        response = client.models.generate_content(
            model="gemini-3-pro-image-preview",
            contents=[*input_images, text_input],
            config=types.GenerateContentConfig(
                safety_settings=safety_settings,
                response_modalities=["TEXT", "IMAGE"],
                image_config=types.ImageConfig(
                    aspect_ratio=aspect_ratio,
                    image_size=resolution,
                ),
            ),
        )

        image_parts: list[bytes] = []

        # Check if we have candidates
        if hasattr(response, "candidates") and response.candidates:
            for part in response.candidates[0].content.parts:
                if hasattr(part, "inline_data") and part.inline_data:
                    image_parts.append(part.inline_data.data)

        # Pull any text for debugging (useful when no image is returned)
        text_bits: list[str] = []
        if hasattr(response, "candidates") and response.candidates:
            for part in response.candidates[0].content.parts:
                if hasattr(part, "text") and part.text:
                    text_bits.append(part.text)
        last_text_debug = "\n".join(t.strip() for t in text_bits if t and t.strip())

        if image_parts:
            image_bytes = image_parts[0]
            image = Image.open(BytesIO(image_bytes))
            image.save(out_path)

            if not out_path.exists():
                raise RuntimeError(f"Save failed: output file not found after save: '{out_path}'")

            # Print full base64 string (not truncated) for full information
            b64_str = base64.b64encode(image_bytes).decode("utf-8")
            print(f"\nImage (base64, PNG):\n{b64_str}\n")
            print(f"✅ Image saved to: {out_path}")
            print("=" * 60)
            return str(out_path)

        # No image returned; retry unless we're out of attempts
        msg = "⚠️ No image generated."
        if last_text_debug:
            msg += f" Model text (preview): {last_text_debug[:400]}"
        print(msg)

        if attempt < max_retries:
            print(f"🔁 Retrying ({attempt}/{max_retries}) after {retry_sleep_s}s...")
            time.sleep(retry_sleep_s)
        else:
            raise RuntimeError(
                f"No image generated after {max_retries} attempts for task '{task.get('name', 'Unknown')}'. "
                + (f"Last model text: {last_text_debug[:800]}" if last_text_debug else "")
            )

In [24]:
# ============================================================================
# STEP 3: Collect ALL GENERATION TASKS (Characters, Cover, Pages)
# ============================================================================
print("\n🎨 Collecting all generation tasks (characters, cover, pages)...")
print("="*70)

# Prepare a unified list of all generation tasks
generation_tasks = []

# --- Characters ---
main_char = story['characters']['main_character']
generation_tasks.append({
    'type': 'character',
    'name': f"Main Character ({main_char.get('name', 'Unknown')})",
    'prompt': main_char['prompt'],
    'input_images': main_char['input_images'],
    'output_image': main_char['output_image']
})

for i, char in enumerate(story['characters'].get('supporting_characters', []), 1):
    generation_tasks.append({
        'type': 'character',
        'name': f"Supporting Character {i} ({char.get('name', 'Unknown')})",
        'prompt': char['prompt'],
        'input_images': char.get('input_images', []),  # Supporting chars may not have input_images
        'output_image': char['output_image']
    })

# --- Book Cover ---
if 'book' in story:
    book = story['book']
    generation_tasks.append({
        'type': 'cover',
        'name': f"Book Cover ({book.get('title', 'Untitled')})",
        'prompt': book['prompt'],
        'input_images': book['input_images'],
        'output_image': book['output_image']
    })

# --- Pages ---
if 'pages' in story:
    for page in story['pages']:
        generation_tasks.append({
            'type': 'page',
            'name': f"Page {page.get('page_number', '?')}",
            'prompt': page['prompt'],
            'input_images': page['input_images'],
            'output_image': page['output_image']
        })

print(f"📊 Total generation tasks: {len(generation_tasks)}\n")


🎨 Collecting all generation tasks (characters, cover, pages)...
📊 Total generation tasks: 13



In [25]:
failures = 0

for idx, task in enumerate(generation_tasks, 1):
    print(
        f"\n--- Generating content for task {idx}/{len(generation_tasks)}: {task.get('name', 'Unknown')} ---"
    )
    try:
        # Use imggen.image_generator (imported in Cell 0) for stronger identity preservation + Avatar-style look
        res = image_generator(
            prompt=task["prompt"],
            image_filenames=task.get("input_images", []) or [],
            output_filename=task["output_image"],
        )
        saved = (res.get("images") or [None])[0]
        print(f"✅ Saved: {saved}")
        print(f"--- Finished generating content for task {idx}/{len(generation_tasks)} ---\n")
    except Exception as e:
        failures += 1
        print(f"\n❌ FAILED task {idx}/{len(generation_tasks)}: {task.get('name', 'Unknown')}")
        print(f"   Error: {e}")
        print("🛑 Stopping generation because later tasks often depend on this output.")
        break

if failures == 0:
    print("\n✅ All generation tasks completed successfully.")

INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.



--- Generating content for task 1/13: Main Character (Agent Babu) ---


INFO:imggen:Loaded reference image: input_images\babu.jpeg
INFO:imggen:Added: 'Reference Face Photo (use this exact face):' -> babu.jpeg
INFO:imggen:Added prompt (586 chars)
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/imgstr/locations/global/publishers/google/models/gemini-3-pro-image-preview:generateContent "HTTP/1.1 200 OK"
INFO:imggen:Response received from model: gemini-3-pro-image-preview
INFO:imggen:Found 4 parts in response
INFO:imggen:Processing part 1/4
INFO:imggen:🧠 Model Thought: **Defining the Subject's Look**

I'm focused on capturing the subject's key features: dark skin, a prominent mustache, and a goatee. The facial structure is also being carefully considered. The aim is...
INFO:imggen:Processing part 2/4
INFO:imggen:🧠 Model Thought: **Constructing the Scene**

I'm now detailing the setting. The focus is on a neutral background to isolate the subject. The lighting wi

✅ Saved: generated\babu_agent.png
--- Finished generating content for task 1/13 ---


--- Generating content for task 2/13: Supporting Character 1 (Agent NTR) ---


INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/imgstr/locations/global/publishers/google/models/gemini-3-pro-image-preview:generateContent "HTTP/1.1 200 OK"
INFO:imggen:Response received from model: gemini-3-pro-image-preview
INFO:imggen:Found 5 parts in response
INFO:imggen:Processing part 1/5
INFO:imggen:🧠 Model Thought: **Examining the Core Task**

I'm focused on the subject matter now, considering the specific details requested to render the Indian action hero effectively. I'm breaking down the elements required for...
INFO:imggen:Processing part 2/5
INFO:imggen:🧠 Model Thought: **Structuring the Prompt Elements**

I'm now carefully integrating the user's requirements into a detailed prompt structure. The goal is to ensure all elements - the subject's features, attire, props,...
INFO:imggen:Processing part 3/5
INFO:imggen:🧠 Model Thought: **Reviewing Image Fidelity**

The recent images seem consistent, matching the original request in many ways. I'

...

✅ Saved: generated\page_8.png
--- Finished generating content for task 11/13 ---


--- Generating content for task 12/13: Page 9 ---


INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/imgstr/locations/global/publishers/google/models/gemini-3-pro-image-preview:generateContent "HTTP/1.1 200 OK"
INFO:imggen:Response received from model: gemini-3-pro-image-preview
INFO:imggen:Found 5 parts in response
INFO:imggen:Processing part 1/5
INFO:imggen:🧠 Model Thought: **Envisioning the Scene**

I am currently focusing on the initial subject: the man himself. I'm considering how to best capture his specific features to ensure the photorealistic quality. The lighting...
INFO:imggen:Processing part 2/5
INFO:imggen:🧠 Model Thought: **Refining the Details**

I'm now prioritizing the man's pose and how he interacts with the gadget. The composition needs to highlight the futuristic nature of the vault and how it interacts with the ...
INFO:imggen:Processing part 3/5
INFO:imggen:🧠 Model Thought: **Comparing the Results**

I'm now carefully examining the generated image, specifically against the initial us

✅ Saved: generated\page_9.png
--- Finished generating content for task 12/13 ---


--- Generating content for task 13/13: Page 10 ---


INFO:imggen:Loaded reference image: generated\jr_ntr.png
INFO:imggen:Added: 'Reference Face Photo (use this exact face):' -> babu.jpeg
INFO:imggen:Added: 'Character Costume (Babu Agent - copy this outfit and hair):' -> babu_agent.png
INFO:imggen:Added: 'Supporting Character (Jr Ntr):' -> jr_ntr.png
INFO:imggen:Added prompt (682 chars)
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/imgstr/locations/global/publishers/google/models/gemini-3-pro-image-preview:generateContent "HTTP/1.1 200 OK"
INFO:imggen:Response received from model: gemini-3-pro-image-preview
INFO:imggen:Found 5 parts in response
INFO:imggen:Processing part 1/5
INFO:imggen:🧠 Model Thought: **Analyzing Character and Attire**

I'm now focusing on the main character's features, specifically replicating the face from a provided image. I will then ensure the character is wearing the appropri...
INFO:imggen:Processing part 2/5
I

✅ Saved: generated\page_10.png
--- Finished generating content for task 13/13 ---


✅ All generation tasks completed successfully.


In [26]:
# ============================================================================
# STEP 7: Generate Premium HTML Storybook
# ============================================================================
print(f"\n📚 Generating Interactive HTML Storybook...")
print("="*70)

html_start = time.time()
try:
    # Fix for Jupyter/IPython: sys.stdout may not have 'reconfigure'
    import sys
    if sys.platform == 'win32':
        try:
            sys.stdout.reconfigure(encoding='utf-8')
        except AttributeError:
            # Jupyter's sys.stdout (OutStream) does not support reconfigure; ignore
            pass

    from create_storybook_html import create_storybook_html

    create_storybook_html(
        json_path="story_data.json",
        output_path="storybook.html",
        images_dir="generated_images"
    )
    timing['html'] = time.time() - html_start

    print(f"\n🎉 SUCCESS! Your premium storybook is ready!")
    print(f"📂 Open 'storybook.html' in your browser to read")
    print(f"⏱️  HTML generation: {timing['html']:.2f} seconds")
    print(f"✨ Features:")
    print(f"   - 📖 3D page-flip animations")
    print(f"   - 📱 Mobile & desktop responsive")
    print(f"   - 🌓 Dark mode toggle")
    print(f"   - ⌨️  Keyboard navigation (Arrow keys)")
    print(f"   - 👆 Touch swipe support")
    print(f"   - 📦 Single file - easy to share!")

except Exception as e:
    timing['html'] = time.time() - html_start
    print(f"⚠️  Could not generate HTML storybook: {str(e)[:200]}")

print(f"\n{'='*70}")
print(f"🎊 ALL DONE! Enjoy your personalized storybook! 🎊")
print(f"{'='*70}")


📚 Generating Interactive HTML Storybook...
📖 Loading story data...
🎨 Embedding images...
   Page 1...
   Page 2...
   Page 3...
   Page 4...
   Page 5...
   Page 6...
   Page 7...
   Page 8...
   Page 9...
   Page 10...
⚡ Generating neumorphism HTML...

✅ Neumorphism storybook created!
📄 File: storybook.html
📦 Size: 3.09 MB

🎯 Features:
   ✓ Beautiful neumorphic design
   ✓ Split-layout (image + text)
   ✓ Animated shimmer title
   ✓ Image lightbox on click
   ✓ Auto-hide navigation (4s timeout)
   ✓ Home button to return to cover
   ✓ Smooth page fade transitions
   ✓ Progress indicator
   ✓ Fullscreen mode on start
   ✓ Keyboard navigation (← → Space Esc)
   ✓ Touch swipe support
   ✓ Mobile responsive
   ✓ Single file - easy to share!

🎉 SUCCESS! Your premium storybook is ready!
📂 Open 'storybook.html' in your browser to read
⏱️  HTML generation: 2.77 seconds
✨ Features:
   - 📖 3D page-flip animations
   - 📱 Mobile & desktop responsive
   - 🌓 Dark mode toggle
   - ⌨️  Keyboard navi